#**Reproducible Image Classification Pipeline with Data Versioning and Automated Reporting**

## Mount Google Drive & Setup Folders

In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a project directory in your Drive
PROJECT_DIR = '/content/drive/MyDrive/SEAI_Image_Pipeline'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Change working directory to the project folder
%cd {PROJECT_DIR}

MessageError: Error: credential propagation was unsuccessful

## Install and Initialize DVC

In [ ]:
# Install DVC
!pip install dvc dvc-gdrive -q

# Initialize Git and DVC in the project folder
!git init -q
!dvc init -q

# Create a directory for our data
!mkdir -p data

## Download Data and Track with DVC

In [ ]:
import torchvision
import torch

# Download the dataset
print("Downloading CIFAR-10 Dataset...")
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)

# Now, track the data folder with DVC using shell commands
!dvc add data
!git add data.dvc .gitignore
!git commit -m "Add raw CIFAR-10 data"

## The Training Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import json
import random
import numpy as np

# 1. ENFORCE REPRODUCIBILITY (Strict Seeding)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in PyTorch
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. DATA PREPARATION
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

# 3. DEFINE A SIMPLE CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 16 * 16, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(-1, 16 * 16 * 16)
        x = self.fc1(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. TRAINING LOOP (Limited to 2 epochs for quick testing)
epochs = 2
loss_history = []

print("Starting Training Pipeline...")
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(trainloader)
    loss_history.append(avg_loss)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

# 5. SAVE METRICS FOR AUTOMATED REPORTING
metrics = {"final_loss": loss_history[-1], "epochs_trained": epochs}
with open("metrics.json", "w") as f:
    json.dump(metrics, f)

print("Pipeline Complete. Metrics saved.")

## Generate the Automated HTML Report

In [ ]:
# Install nbconvert
!pip install nbconvert -q

# Colab stores a copy of the notebook you are currently working on in your Drive
# (assuming you created it in Drive). We will convert it to HTML.
# IMPORTANT: Replace 'Untitled0.ipynb' with the actual name of your notebook!

!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb" --output-dir {PROJECT_DIR}

print("Automated Report Generated Successfully!")